In [33]:
# Import Necessary Packages/Modules
import time, numpy as np
import pandas as pd
from skeLCS import eLCS # Run "pip install scikit-eLCS" to install necessary module
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import balanced_accuracy_score

In [34]:
# Define the leakage columns again
LEAKAGE_COLS = ["is_high_risk","annual_premium","monthly_premium","annual_medical_cost",
                "claims_count","avg_claim_amount","total_claims_paid"]

# Read file into a table. Turns off default behavior that reads "none" as missing
df = pd.read_csv("medical_insurance.csv", keep_default_na=False, na_values=[""])

In [35]:
# Documenting every issue before fixing it
print("shape:", df.shape)
print("duplicates:", df.duplicated().sum(), "| nulls:", df.isna().sum().sum())
print("negative values anywhere:", bool((df.select_dtypes("number") < 0).any().any()))

# Age profile
print("\nage range:", df["age"].min(), "-", df["age"].max())
print("age <= 1 count:", (df["age"] <= 1).sum())
print(df.loc[df["age"] < 18, "age"].value_counts().sort_index())

# Implausible BMI
print("\nbmi < 13 count:", (df["bmi"] < 13).sum(), "| bmi > 70 count:", (df["bmi"] > 70).sum())
print("infant AND bmi<13 overlap:", ((df["age"] <= 1) & (df["bmi"] < 13)).sum())

# risk_score clipping
print("\nrisk_score == 0:", (df["risk_score"] == 0).sum(), "| risk_score == 1:", (df["risk_score"] == 1).sum())

# Category spellings
print()
for c in df.select_dtypes(exclude="number"):
    print(c, sorted(df[c].unique()))

shape: (100000, 54)
duplicates: 0 | nulls: 0
negative values anywhere: False

age range: 0 - 100
age <= 1 count: 214
age
0     165
1      49
2      37
3      42
4      61
5      85
6      91
7     107
8     129
9     130
10    159
11    170
12    212
13    234
14    280
15    329
16    365
17    399
Name: count, dtype: int64

bmi < 13 count: 250 | bmi > 70 count: 0
infant AND bmi<13 overlap: 1

risk_score == 0: 1131 | risk_score == 1: 5241

sex ['Female', 'Male', 'Other']
region ['Central', 'East', 'North', 'South', 'West']
urban_rural ['Rural', 'Suburban', 'Urban']
education ['Bachelors', 'Doctorate', 'HS', 'Masters', 'No HS', 'Some College']
marital_status ['Divorced', 'Married', 'Single', 'Widowed']
employment_status ['Employed', 'Retired', 'Self-employed', 'Unemployed']
smoker ['Current', 'Former', 'Never']
alcohol_freq ['Daily', 'None', 'Occasional', 'Weekly']
plan_type ['EPO', 'HMO', 'POS', 'PPO']
network_tier ['Bronze', 'Gold', 'Platinum', 'Silver']


In [36]:
# Is the child population consistent with an adult insurance product
minors = df[df["age"] < 18]
adults = df[df["age"] >= 18]
print(f"minors: {len(minors)} ({len(minors)/len(df):.2%}) | adults: {len(adults)}")
print("mean risk_score  - minors:", round(minors["risk_score"].mean(), 3), "| adults:", round(adults["risk_score"].mean(), 3))
print("mean chronic_count - minors:", round(minors["chronic_count"].mean(), 3), "| adults:", round(adults["chronic_count"].mean(), 3))
print("smoker shares - minors:", minors["smoker"].value_counts(normalize=True).round(3).to_dict())
print("smoker shares - adults:", adults["smoker"].value_counts(normalize=True).round(3).to_dict())

minors: 3044 (3.04%) | adults: 96956
mean risk_score  - minors: 0.183 | adults: 0.53
mean chronic_count - minors: 0.622 | adults: 0.728
smoker shares - minors: {'Never': 0.685, 'Former': 0.179, 'Current': 0.136}
smoker shares - adults: {'Never': 0.697, 'Former': 0.182, 'Current': 0.121}


In [37]:
# Cleaning decisions. Applied before splitting, so no leakage is possible
MIN_AGE = 18 # Restrict the data to only adults. I don't trust the data saying that minors smoke at the same current rate as adults, seems like erronious data
DROP_BAD_BMI = True  # Set False to keep implausible BMI instead

n0 = len(df)
df = df.drop_duplicates() # Phase-1 found 0, but I'd like to show that I am trying to account for it regardless

# Minors are outside the adult population that would be the primary focus of our study
# A BMI of less than 13 is not plausable in a living human and thus should not be considered for this purpose
minor_mask = df["age"] < MIN_AGE
bad_bmi_mask = (df["bmi"] < 13) if DROP_BAD_BMI else None

# Marks only rows that are infants for removal if there are no bad BMI scores, else it'll mark either infants or have a bad BMI
removed = minor_mask if bad_bmi_mask is None else (minor_mask | bad_bmi_mask)

print(f"removing: minors={minor_mask.sum()}, implausible BMI={(bad_bmi_mask.sum() if bad_bmi_mask is not None else 0)}, "
      f"overlap={((minor_mask & (df['bmi'] < 13)).sum())}, total={removed.sum()}")
df = df[~removed].reset_index(drop=True)
print(f"records: {n0} -> {len(df)} ({(n0 - len(df)) / n0:.2%} removed)")

# Check that nothing flagged survived, just in case
assert (df["age"] <= 1).sum() == 0
if DROP_BAD_BMI:
    assert (df["bmi"] < 13).sum() == 0

removing: minors=3044, implausible BMI=250, overlap=7, total=3287
records: 100000 -> 96713 (3.29% removed)


In [38]:
# Save the cleaned dataset so every later task uses this file specifically, for repeatability
df.to_csv("medical_insurance_cleaned_v2.csv", index=False)
print("saved medical_insurance_cleaned_v2.csv", df.shape)

saved medical_insurance_cleaned_v2.csv (96713, 54)


In [39]:
# Making rough thirds of risk score for stratify to work from
tmp_band = pd.qcut(df["risk_score"], 3, labels=False)
# Splitting 80% for training, 20% for testing and seeding every test for repeatable results
train, test = train_test_split(df, test_size=0.2, stratify=tmp_band, random_state=42)

In [40]:
# eLCS predicts classes, so we're cutting risk_score into 3 groups
q = train["risk_score"].quantile([1/3, 2/3]).values
edges = [-np.inf, q[0], q[1], np.inf] # The np.inf is just saying no upper or lower limits
y_train = pd.cut(train["risk_score"], edges, labels=False).values
y_test  = pd.cut(test["risk_score"],  edges, labels=False).values
print("band edges:", q) # The train's quantiles
print("class shares:", np.round(np.bincount(y_train) / len(y_train), 3)) # Imbalance check

band edges: [0.4066 0.6374]
class shares: [0.35  0.328 0.322]


In [41]:
# Quality checks
print("duplicates:", df.duplicated().sum(), "| nulls:", df.isna().sum().sum())

duplicates: 0 | nulls: 0


In [42]:
# Data-quality and outlier evidence
print("column types:", train.dtypes.astype(str).value_counts().to_dict())
for c in train.select_dtypes(exclude="number"):
    print(c, sorted(train[c].unique())) # looks for spelling or case variants
print("negative values anywhere:", bool((train.select_dtypes("number") < 0).any().any()))

infants = (train["age"] <= 1).values
low_bmi = (train["bmi"] < 13).values
print("infants:", infants.sum(), np.round(np.bincount(y_train[infants], minlength=3) / infants.sum(), 2))
print("BMI < 13:", low_bmi.sum(), np.round(np.bincount(y_train[low_bmi], minlength=3) / low_bmi.sum(), 2))

column types: {'int64': 31, 'float64': 13, 'str': 10}
sex ['Female', 'Male', 'Other']
region ['Central', 'East', 'North', 'South', 'West']
urban_rural ['Rural', 'Suburban', 'Urban']
education ['Bachelors', 'Doctorate', 'HS', 'Masters', 'No HS', 'Some College']
marital_status ['Divorced', 'Married', 'Single', 'Widowed']
employment_status ['Employed', 'Retired', 'Self-employed', 'Unemployed']
smoker ['Current', 'Former', 'Never']
alcohol_freq ['Daily', 'None', 'Occasional', 'Weekly']
plan_type ['EPO', 'HMO', 'POS', 'PPO']
network_tier ['Bronze', 'Gold', 'Platinum', 'Silver']
negative values anywhere: False
infants: 0 [nan nan nan]
BMI < 13: 0 [nan nan nan]


C:\Users\devme\AppData\Local\Temp\ipykernel_23516\4036522088.py:9: RuntimeWarning: invalid value encountered in divide
  print("infants:", infants.sum(), np.round(np.bincount(y_train[infants], minlength=3) / infants.sum(), 2))
C:\Users\devme\AppData\Local\Temp\ipykernel_23516\4036522088.py:10: RuntimeWarning: invalid value encountered in divide
  print("BMI < 13:", low_bmi.sum(), np.round(np.bincount(y_train[low_bmi], minlength=3) / low_bmi.sum(), 2))


In [43]:
# Drop leaky and ID columns, and encode text as integer codes
def prepare(d):
    X = d.drop(columns=LEAKAGE_COLS + ["risk_score", "person_id"]).copy()
    for c in X.select_dtypes(exclude="number"):
        X[c] = X[c].astype("category").cat.codes
    return X

X_train, X_test = prepare(train), prepare(test)

In [44]:
# Feature selection on the training set only
tree = DecisionTreeClassifier(max_depth=5, random_state=42).fit(X_train, y_train) # A shallow tree finds which columns separate the bands
importance = pd.Series(tree.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print(importance.head(6).round(3))
selected = list(importance.index[:4])
print("selected:", selected)

chronic_count    0.420
age              0.345
smoker           0.169
bmi              0.066
sex              0.000
income           0.000
dtype: float64
selected: ['chronic_count', 'age', 'smoker', 'bmi']


In [45]:
# All Features vs top 4
def run(name, cols):
    t0 = time.time()
    m = eLCS(random_state=42).fit(X_train[cols].values.astype(float), y_train)
    s = balanced_accuracy_score(y_test, m.predict(X_test[cols].values.astype(float)))
    print(f"{name}: test balanced accuracy {s:.3f} | {time.time()-t0:.0f}s")

run("Before (45 features)", list(X_train.columns))
run("After (4 selected)", selected)

Before (45 features): test balanced accuracy 0.569 | 95s
After (4 selected): test balanced accuracy 0.856 | 24s
